In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import pandas as pd
import numpy as np
from pandas import DataFrame, merge, concat

from numpy import floor, ceil, cumsum, where
from collections import defaultdict
from logging import getLogger
import math, time


## Classes

from objects.Axle import Axle
from objects.Container import Container
from objects.ContainerSummary import ContainerSummary
from objects.ContainerLoadingRules import ContainerLoadingRules
from objects.Dimension import Dimension
from objects.Pallet import Pallet
from objects.Position import Position
import uuid

In [4]:
logger = getLogger("load_planner")

# Assumptions
1. Units in the pallet are homogeneous
2. Units are calculated from item quantity to pallets
3. Dimensions are measured in inches (Later converted to Feet / other metrics)
4. **Routes have been pre-planned**
5. 

# Solver
1. Decide what items to load based on:
- Delivery date, priority, Item name, 
- Convert units into pallets
- Confirm the #of units shipped & update 
2. Grouping logic:
- Combine all items

### Optimizers to look into
1. 
## Feature enhancements
1. Stock pull-in from future (Early shipping)
2. Axle based handling-unit 📦(container) positioning
3. 

In [5]:
### Writing results to Database ###
# Write results to tables:
# 1. Handling_unit
# 2. Handling_unit_content
# 3. Handling_unit_position
# 4. Transport_equipment_assignment
# 5. route_planned


In [6]:
### Verify loaded 🚚 truck_equipment_assignment & handling_unit positions inside the container  

## Establish connection with Neon Database

In [7]:
### NeonDB Connection
import database.helper as db_helper
db_conn = db_helper.create_connection()

## Read Data From Database

In [8]:
item_master_df = db_helper.fetch_data(sql="select * from inventory_management.public.item_master", connection=db_conn)
lane_master_df = db_helper.fetch_data(sql="select * from inventory_management.public.lane_master", connection=db_conn)
load_equipment_metadata_df = db_helper.fetch_data(sql="select * from inventory_management.public.load_equipment_metadata", connection=db_conn)
location_df = db_helper.fetch_data(sql="select * from inventory_management.public.location", connection=db_conn)
shipment_plans_df = db_helper.fetch_data(sql="select * from inventory_management.public.shipment_plans", connection=db_conn)
sku_uom_df = db_helper.fetch_data(sql="select * from inventory_management.public.sku_unit_of_measure", connection=db_conn)
transport_asset_df = db_helper.fetch_data(sql="select * from inventory_management.public.transport_asset", connection=db_conn)


S:\git_repo\solutions-inventory-optimization\database\helper.py:39: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(


## Create necessary features, calculations




In [9]:
def container_visualization(data:Container={}):
    from IPython.display import IFrame, display
    import json
    import urllib.parse
    encoded = ""
    if data:
        json_string = json.dumps(data)
        encoded = urllib.parse.quote(json_string)
    container_iframe = IFrame(
        src="http://localhost:5173/container-visualization" f"?data={encoded}",
        width="100%",
        height=450
    )

    display(container_iframe)

In [10]:
def mm_to_cm(series):
    """
    Convert millimeters to centimeters.

    Input:
        Pandas Series

    Returns:
        Pandas Series
    """

    return series.astype(float) / 10

In [11]:
def calculate_volume_m3(
    length_mm,
    width_mm,
    height_mm
):
    """
    Calculate cubic meters from
    millimeter dimensions.

    Accepts Pandas Series.
    """

    return (
        length_mm.astype(float)
        * width_mm.astype(float)
        * height_mm.astype(float)
    ) / 1_000_000_000

In [12]:
def mm3_to_m3(volume_mm3):

    return volume_mm3.astype(float) / 1_000_000_000

In [13]:
def calculate_pallet_floor_area_m2(
    length_mm,
    width_mm
):
    return (
        length_mm * width_mm
    ) / 1_000_000

In [14]:
sku_uom_df = pd.concat(
    [
        sku_uom_df,
        sku_uom_df['pallet_dimensions'].apply(pd.Series)
    ],
    axis=1
)

sku_uom_column_mapper = {x:x for x in sku_uom_df.columns}
sku_uom_column_mapper['height_mm'] = 'pallet_height_mm'
sku_uom_column_mapper['width_mm'] = 'pallet_width_mm'
sku_uom_column_mapper['length_mm'] = 'pallet_length_mm'
sku_uom_df.rename(columns=sku_uom_column_mapper, inplace=True)

In [15]:


sku_uom_df['sku_id'] = sku_uom_df['sku_id'].astype(str)
shipment_plans_df['sku_id'] = shipment_plans_df['sku_id'].astype(str)

In [16]:
shipment_plans_df = pd.merge(
    left=shipment_plans_df, 
    right=sku_uom_df[['sku_id', 'unit_count_in_pallet', 'pallet_height_mm', 'pallet_width_mm', 'pallet_length_mm', ]], 
    left_on=['sku_id'], 
    right_on=['sku_id'], 
    how='left'
)


In [17]:

shipment_plans_df['planned_quantity_units_per_pallet'] = shipment_plans_df['planned_quantity'] / shipment_plans_df['unit_count_in_pallet']

shipment_plans_df['item_weight_kg'] = (shipment_plans_df['weight_kg']/shipment_plans_df['planned_quantity']).round(2)


# ----------------------------------
# Remaining Quantity
# ----------------------------------

shipment_plans_df["remaining_quantity"] = (
    shipment_plans_df["planned_quantity"]
    -
    shipment_plans_df["shipped_quantity"]
    .fillna(0)
)


In [18]:

# ----------------------------------
# Full Pallets
# ----------------------------------

shipment_plans_df["full_pallet_count"] = (
    shipment_plans_df["remaining_quantity"]
    //
    shipment_plans_df["unit_count_in_pallet"]
)

# ----------------------------------
# Remaining Units
# ----------------------------------

shipment_plans_df["remaining_units"] = (
    shipment_plans_df["remaining_quantity"]
    %
    shipment_plans_df["unit_count_in_pallet"]
)

# ----------------------------------
# Partial Pallet Fill %
# ----------------------------------

shipment_plans_df["pallet_fill_pct"] = np.where(
    shipment_plans_df["unit_count_in_pallet"] > 0,
    shipment_plans_df["remaining_units"]
    /
    shipment_plans_df["unit_count_in_pallet"],
    0
)


## Phase 1:



In [19]:
def build_fixed_container(
	load_equipment_metadata_df
):
	"""
	Build fixed 40FT container.
	
	Returns
	-------
	Container
	"""

	equipment_row = (

		load_equipment_metadata_df

		.loc[
			load_equipment_metadata_df[
				"equipment_name"
			]

			.str.upper()

			.str.contains(
				"40FT",
				na=False
			)
		]

		.iloc[0]
	)

	container = Container(

		containerId=str(
			equipment_row["equipment_id"]
		),

		containerType=
			equipment_row["equipment_name"],

		length=
			equipment_row["length_mm"],

		width=
			equipment_row["width_mm"],

		height=
			equipment_row["height_mm"],

		internal_length=
			equipment_row["internal_length_mm"],

		internal_width=
			equipment_row["internal_width_mm"],

		internal_height=
			equipment_row["internal_height_mm"],

		maxPayloadWeight=
			equipment_row["max_payload_weight_kg"],

		tareWeight=
			equipment_row["tare_weight_kg"],

		maxVolume=(

			equipment_row["internal_length_mm"]
			*
			equipment_row["internal_width_mm"]
			*
			equipment_row["internal_height_mm"]

		) / 1_000_000_000,

		door_width=
			equipment_row["door_width_mm"],

		door_height=
			equipment_row["door_height_mm"],

		pallets=[]
	)

	return container


In [20]:
container = build_fixed_container(load_equipment_metadata_df=load_equipment_metadata_df)

In [21]:
# Step 1 - Build Daily Demand
def build_daily_demand(
    shipment_plans_df: pd.DataFrame,
    planning_date
):

    df = shipment_plans_df.copy()

    planning_date = pd.to_datetime(
        planning_date
    )

    df["estimated_delivery_date"] = pd.to_datetime(
        df["estimated_delivery_date"],
        errors="coerce"
    )

    df = df[
        df["estimated_delivery_date"]
        <= planning_date
    ]

    df = df[
        df["planned_quantity"] > 0
    ]

    df["remaining_quantity"] = (

        df["planned_quantity"]

        -

        df["shipped_quantity"].fillna(0)
    )

    df = df[
        df["remaining_quantity"] > 0
    ]

    return df.reset_index(
        drop=True
    )

In [22]:
# Step 2 - Convert Units To Pallets

def convert_units_to_pallets(
    daily_demand_df
):

    df = daily_demand_df.copy()

    df["required_pallets"] = (

        df["remaining_quantity"]

        /

        df["unit_count_in_pallet"]
    )

    return df

In [23]:
# Step 3 - Apply Partial Pallet Rules

def apply_partial_pallet_rules(
    pallet_df,
    round_up_threshold=0.6
):

    df = pallet_df.copy()

    df["full_pallets"] = np.floor(
        df["required_pallets"]
    )

    df["fractional_pallet"] = (

        df["required_pallets"]

        -

        df["full_pallets"]
    )

    df["rounded_pallets"] = np.where(

        df["fractional_pallet"]

        >= round_up_threshold,

        np.ceil(
            df["required_pallets"]
        ),

        np.floor(
            df["required_pallets"]
        )
    )

    return df

In [24]:
# Step 4 - Build Shipment Buckets
def build_shipment_buckets(
    shipment_df
):

    bucket_df = (

        shipment_df

        .groupby(
            [
                "estimated_delivery_date",
                "origin_location_id",
                "destination_location_id",
                "sku_id"
            ],
            as_index=False
        )

        .agg(
            {
                "rounded_pallets": "sum",
                "weight_kg": "sum",
                "priority": "max"
            }
        )
    )

    return bucket_df

In [25]:
# Step 5 - Build Load Queue

def build_load_queue(
    bucket_df
):

    return (

        bucket_df

        .sort_values(
            [
                "estimated_delivery_date",
                "priority",
                "rounded_pallets"
            ],
            ascending=[
                True,
                False,
                False
            ]
        )

        .reset_index(drop=True)
    )

In [26]:
# Step 6 - Build Fixed Container

def build_fixed_container(
    load_equipment_metadata_df
):

    row = (

        load_equipment_metadata_df

        .loc[
            load_equipment_metadata_df[
                "equipment_name"
            ]

            .str.upper()

            .str.contains(
                "40FT",
                na=False
            )
        ]

        .iloc[0]
    )

    return {

        "equipment_id":
            row["equipment_id"],

        "equipment_name":
            row["equipment_name"],

        "internal_length_mm":
            row["internal_length_mm"],

        "internal_width_mm":
            row["internal_width_mm"],

        "internal_height_mm":
            row["internal_height_mm"],

        "max_payload_weight_kg":
            row["max_payload_weight_kg"]
    }

In [27]:
# Step 7 - Container Build Solver

def solve_container_build(
    load_queue_df,
    container
):

    container_builds = []

    container_number = 1

    current_weight = 0

    current_pallets = []

    max_weight = (
        container[
            "max_payload_weight_kg"
        ]
    )

    for _, row in load_queue_df.iterrows():

        pallet_weight = (
            row["weight_kg"]
            /
            max(
                row["rounded_pallets"],
                1
            )
        )

        pallet_count = int(
            row["rounded_pallets"]
        )

        for _ in range(
            pallet_count
        ):

            if (

                current_weight
                + pallet_weight

                >

                max_weight

            ):

                container_builds.append({

                    "container_id":

                        f"CONT_{container_number}",

                    "pallets":

                        current_pallets
                })

                container_number += 1

                current_pallets = []

                current_weight = 0

            current_pallets.append({

                "sku_id":
                    row["sku_id"],

                "weight_kg":
                    pallet_weight,

                "destination_location_id":
                    row["destination_location_id"]
            })

            current_weight += (
                pallet_weight
            )

    if current_pallets:

        container_builds.append({

            "container_id":

                f"CONT_{container_number}",

            "pallets":

                current_pallets
        })

    return container_builds


In [28]:
# Step 8 - Build Physical Pallets

def build_pallets(
    container_builds,
    shipment_plans_df
):

    pallets = []

    for container in container_builds:

        for pallet in container["pallets"]:

            sku = pallet["sku_id"]

            shipment_row = (

                shipment_plans_df

                .loc[
                    shipment_plans_df[
                        "sku_id"
                    ]
                    == sku
                ]

                .iloc[0]
            )

            pallets.append({

                "container_id":
                    container[
                        "container_id"
                    ],

                "pallet_id":

                    str(
                        uuid.uuid4()
                    ),

                "sku_id":
                    sku,

                "weight_kg":
                    pallet[
                        "weight_kg"
                    ],

                "length_mm":

                    shipment_row[
                        "pallet_length_mm"
                    ],

                "width_mm":

                    shipment_row[
                        "pallet_width_mm"
                    ],

                "height_mm":

                    shipment_row[
                        "pallet_height_mm"
                    ],

                "position_x": 0,

                "position_y": 0,

                "position_z": 0
            })

    return pd.DataFrame(
        pallets
    )




In [29]:
# Step 9 - Place Pallets


def place_pallets(
    pallet_df,
    container
):

    df = pallet_df.copy()

    current_x = 0
    current_z = 0

    row_depth = 0

    container_length = (
        container[
            "internal_length_mm"
        ]
    )

    container_width = (
        container[
            "internal_width_mm"
        ]
    )

    for idx in df.index:

        pallet_length = (
            df.loc[
                idx,
                "length_mm"
            ]
        )

        pallet_width = (
            df.loc[
                idx,
                "width_mm"
            ]
        )

        if (

            current_z
            + pallet_width

            >

            container_width

        ):

            current_x += row_depth

            current_z = 0

            row_depth = 0

        if (

            current_x
            + pallet_length

            >

            container_length

        ):

            raise Exception(
                "Container full"
            )

        df.loc[
            idx,
            "position_x"
        ] = current_x

        df.loc[
            idx,
            "position_z"
        ] = current_z

        current_z += pallet_width

        row_depth = max(
            row_depth,
            pallet_length
        )

    return df

In [30]:
# Step 10 - Center of Gravity

def calculate_center_of_gravity(
    pallet_df
):

    total_weight = (
        pallet_df[
            "weight_kg"
        ].sum()
    )

    cg_x = (

        pallet_df[
            "weight_kg"
        ]

        *

        pallet_df[
            "position_x"
        ]

    ).sum() / total_weight

    cg_z = (

        pallet_df[
            "weight_kg"
        ]

        *

        pallet_df[
            "position_z"
        ]

    ).sum() / total_weight

    return {

        "cg_x": cg_x,

        "cg_z": cg_z
    }

In [ ]:
# Run Pipeline

planning_date = "2026-06-10"

daily_demand_df = build_daily_demand(
    shipment_plans_df,
    planning_date
)

pallet_req_df = convert_units_to_pallets(
    daily_demand_df
)

shipment_bucket_df = (
    apply_partial_pallet_rules(
        pallet_req_df
    )
)

bucket_df = build_shipment_buckets(
    shipment_bucket_df
)

load_queue_df = build_load_queue(
    bucket_df
)

container = build_fixed_container(
    load_equipment_metadata_df
)

container_builds = (
    solve_container_build(
        load_queue_df,
        container
    )
)

pallet_df = build_pallets(
    container_builds,
    shipment_plans_df
)

pallet_df = place_pallets(
    pallet_df,
    container
)

cg = calculate_center_of_gravity(
    pallet_df
)

print(cg)


## Create Links for the following
1. Transport equipment assignment 🚚
   1. (transport_asset_df + load_equipment) [🛻+📦 record] 
2. 